In [19]:
import openml
import pandas as pd
import numpy as np
import os
# Load oil dataset from OpenML
oil = openml.datasets.get_dataset(311)
X_oil, y_oil, *_ = oil.get_data(dataset_format="dataframe", target=oil.default_target_attribute)

# Load Kaggle Credit Card Fraud Detection dataset
df_fraud_raw = pd.read_csv("creditcard.csv")
# Separate features and target (target column is 'Class')
X_fraud = df_fraud_raw.drop('Class', axis=1)
y_fraud = df_fraud_raw['Class']

In [4]:
# Explore oil dataset
print("=== OIL DATASET ===")
print(f"Dataset name: {oil.name}")
print(f"Shape: {X_oil.shape}")
print(f"\nFeatures ({len(X_oil.columns)}):")
print(X_oil.columns.tolist())
print(f"\nFeature types:")
print(X_oil.dtypes)
print(f"\nTarget variable: {oil.default_target_attribute}")
print(f"Target distribution:\n{y_oil.value_counts()}")
print(f"\nClass imbalance ratio: {y_oil.value_counts().min() / y_oil.value_counts().max():.4f}")

=== OIL DATASET ===
Dataset name: oil_spill
Shape: (937, 49)

Features (49):
['attr1', 'attr2', 'attr3', 'attr4', 'attr5', 'attr6', 'attr7', 'attr8', 'attr9', 'attr10', 'attr11', 'attr12', 'attr13', 'attr14', 'attr15', 'attr16', 'attr17', 'attr18', 'attr19', 'attr20', 'attr21', 'attr22', 'attr23', 'attr24', 'attr25', 'attr26', 'attr27', 'attr28', 'attr29', 'attr30', 'attr31', 'attr32', 'attr33', 'attr34', 'attr35', 'attr36', 'attr37', 'attr38', 'attr39', 'attr40', 'attr41', 'attr42', 'attr43', 'attr44', 'attr45', 'attr46', 'attr47', 'attr48', 'attr49']

Feature types:
attr1     float64
attr2     float64
attr3     float64
attr4     float64
attr5       uint8
attr6     float64
attr7     float64
attr8     float64
attr9     float64
attr10    float64
attr11    float64
attr12    float64
attr13    float64
attr14    float64
attr15    float64
attr16    float64
attr17    float64
attr18    float64
attr19    float64
attr20    float64
attr21    float64
attr22    float64
attr23      uint8
attr24    f

In [ ]:
# CORRECTED VERSION: Prepare datasets for k-center matching
# Replace the function in the cell above with this corrected version

def prepare_dataset_for_kcenter(X, y, dataset_name):
    """
    Prepare a dataset for k-center matching by:
    1. Creating ENROLID column
    2. Converting target from -1/1 to 0/1 (1 = minority/cases, 0 = majority/controls)
    3. Combining features and target into a single dataframe
    
    This version handles object types and different target formats.
    """
    # Create dataframe with features
    df = X.copy()
    
    # Add ENROLID (using index + 1 as IDs)
    df['ENROLID'] = range(1, len(df) + 1)
    
    # Convert target to numeric if it's not already
    y_numeric = pd.to_numeric(y, errors='coerce')
    
    # Check unique values to determine format
    unique_vals = y_numeric.dropna().unique()
    
    # Convert target based on format:
    # - If values are -1 and 1: convert -1 -> 0, 1 -> 1
    # - If values are already 0 and 1: keep as is
    if set(unique_vals).issubset({-1, 1}):
        # Format is -1/1, convert -1 -> 0, 1 -> 1
        df['target'] = ((y_numeric == 1).astype(int))
    elif set(unique_vals).issubset({0, 1}):
        # Format is already 0/1
        df['target'] = y_numeric.astype(int)
    else:
        # Find the minority class (smaller count) and map it to 1
        value_counts = y_numeric.value_counts()
        minority_value = value_counts.idxmin()
        df['target'] = (y_numeric == minority_value).astype(int)
        print(f"  Note: Mapped minority class value {minority_value} to 1")
    
    # Reorder columns to put ENROLID and target first
    cols = ['ENROLID', 'target'] + [c for c in df.columns if c not in ['ENROLID', 'target']]
    df = df[cols]
    
    print(f"\n=== {dataset_name.upper()} DATASET PREPARED ===")
    print(f"Shape: {df.shape}")
    print(f"Original target type: {type(y)}")
    print(f"Original target unique values: {sorted(unique_vals)}")
    print(f"Target distribution:")
    print(df['target'].value_counts().sort_index())
    print(f"Class imbalance ratio: {df['target'].value_counts().min() / df['target'].value_counts().max():.4f}")
    
    return df

# Use the fixed function
df_oil = prepare_dataset_for_kcenter(X_oil, y_oil, "oil")


=== OIL DATASET PREPARED ===
Shape: (937, 51)
Original target type: <class 'pandas.core.series.Series'>
Original target unique values: [np.int64(-1), np.int64(1)]
Target distribution:
target
0    896
1     41
Name: count, dtype: int64
Class imbalance ratio: 0.0458


In [13]:
# Explore credit card fraud dataset
if X_fraud is not None:
    print("=== CREDIT CARD FRAUD DATASET ===")
    print(f"Shape: {X_fraud.shape}")
    print(f"\nFeatures ({len(X_fraud.columns)}):")
    print(X_fraud.columns.tolist())
    print(f"\nFeature types:")
    print(X_fraud.dtypes.value_counts())
    print(f"\nTarget variable: 'Class'")
    print(f"Target distribution:\n{y_fraud.value_counts()}")
    print(f"\nClass imbalance ratio: {y_fraud.value_counts().min() / y_fraud.value_counts().max():.4f}")
    print(f"\nFirst few rows:")
    X_fraud.head()
    
    # Show some statistics
    print(f"\nDataset statistics:")
    print(f"  Total samples: {len(X_fraud):,}")
    print(f"  Fraud cases (1): {y_fraud.sum():,} ({y_fraud.mean()*100:.2f}%)")
    print(f"  Normal cases (0): {(y_fraud == 0).sum():,} ({(1-y_fraud.mean())*100:.2f}%)")
else:
    print("⚠️  Credit card fraud dataset not loaded. Please download from Kaggle first.")

=== CREDIT CARD FRAUD DATASET ===
Shape: (284807, 30)

Features (30):
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']

Feature types:
float64    30
Name: count, dtype: int64

Target variable: 'Class'
Target distribution:
Class
0    284315
1       492
Name: count, dtype: int64

Class imbalance ratio: 0.0017

First few rows:

Dataset statistics:
  Total samples: 284,807
  Fraud cases (1): 492 (0.17%)
  Normal cases (0): 284,315 (99.83%)


In [14]:
# Setup feature columns and column types (similar to kcenter_hyperparameter_search_global.py)
def setup_feature_columns(df, target_col='target'):
    """Setup feature columns, categorical columns, and numeric columns."""
    
    # Feature columns are all columns except ENROLID and target
    feature_cols = [c for c in df.columns if c not in ['ENROLID', target_col]]
    
    # Identify categorical and numeric columns
    # Check for object/category types
    cat_cols = df[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
    
    # Numeric columns (excluding binary flags)
    numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    
    # Check for binary columns (0/1 or uint8 with only 0/1 values)
    bin_cols = []
    for col in numeric_cols:
        unique_vals = df[col].dropna().unique()
        if len(unique_vals) <= 2 and set(unique_vals).issubset({0, 1, 0.0, 1.0}):
            bin_cols.append(col)
    
    # True numeric columns (exclude binary flags)
    true_num_cols = [c for c in numeric_cols if c not in bin_cols]
    
    print(f"\n=== COLUMN SETUP ===")
    print(f"Total features: {len(feature_cols)}")
    print(f"Categorical columns: {len(cat_cols)}")
    print(f"Binary columns: {len(bin_cols)}")
    print(f"True numeric columns: {len(true_num_cols)}")
    if cat_cols:
        print(f"  Categorical: {cat_cols}")
    if bin_cols:
        print(f"  Binary: {bin_cols[:10]}{'...' if len(bin_cols) > 10 else ''}")
    
    return feature_cols, cat_cols, true_num_cols, bin_cols

# Setup for oil dataset
feature_cols_oil, CAT_COLUMNS_OIL, TRUE_NUM_COLUMNS_OIL, BIN_COLUMNS_OIL = setup_feature_columns(df_oil)
print("\n" + "="*60)

# Setup for credit card fraud dataset
if X_fraud is not None:
    df_fraud = prepare_dataset_for_kcenter(X_fraud, y_fraud, "credit_card_fraud")
    feature_cols_fraud, CAT_COLUMNS_FRAUD, TRUE_NUM_COLUMNS_FRAUD, BIN_COLUMNS_FRAUD = setup_feature_columns(df_fraud)
else:
    print("\n⚠️  Skipping credit card fraud dataset setup (file not loaded)")


=== COLUMN SETUP ===
Total features: 49
Categorical columns: 0
Binary columns: 2
True numeric columns: 47
  Binary: ['attr23', 'attr46']


=== CREDIT_CARD_FRAUD DATASET PREPARED ===
Shape: (284807, 32)
Target distribution:
target
0    284315
1       492
Name: count, dtype: int64
Class imbalance ratio: 0.0017

=== COLUMN SETUP ===
Total features: 30
Categorical columns: 0
Binary columns: 0
True numeric columns: 30


In [15]:
# Train/test splits (similar to kcenter_hyperparameter_search_global.py)
from sklearn.model_selection import train_test_split

TRAIN_TEST_SEED = 123

def create_train_test_split(df, target_col='target', test_size=0.3, val_size=0.5, random_state=123):
    """
    Create train/val/test splits similar to the main script.
    Returns: train_df, val_df, test_df
    """
    # First split: train vs (val+test)
    train_df, temp_df = train_test_split(
        df, 
        test_size=test_size, 
        stratify=df[target_col], 
        random_state=random_state
    )
    
    # Second split: val vs test (from temp_df)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=val_size,
        stratify=temp_df[target_col],
        random_state=random_state
    )
    
    print(f"\n=== DATA SPLITS ===")
    print(f"Train: {len(train_df):,} samples")
    print(f"  - Minority: {(train_df[target_col] == 1).sum():,}")
    print(f"  - Majority: {(train_df[target_col] == 0).sum():,}")
    print(f"Val: {len(val_df):,} samples")
    print(f"  - Minority: {(val_df[target_col] == 1).sum():,}")
    print(f"  - Majority: {(val_df[target_col] == 0).sum():,}")
    print(f"Test: {len(test_df):,} samples")
    print(f"  - Minority: {(test_df[target_col] == 1).sum():,}")
    print(f"  - Majority: {(test_df[target_col] == 0).sum():,}")
    
    return train_df, val_df, test_df

# Create splits for oil dataset
train_oil, val_oil, test_oil = create_train_test_split(df_oil, random_state=TRAIN_TEST_SEED)
print("\n" + "="*60)

# Create splits for credit card fraud dataset
if X_fraud is not None:
    train_fraud, val_fraud, test_fraud = create_train_test_split(df_fraud, random_state=TRAIN_TEST_SEED)
else:
    print("\n⚠️  Skipping credit card fraud dataset splits (file not loaded)")


=== DATA SPLITS ===
Train: 655 samples
  - Minority: 29
  - Majority: 626
Val: 141 samples
  - Minority: 6
  - Majority: 135
Test: 141 samples
  - Minority: 6
  - Majority: 135


=== DATA SPLITS ===
Train: 199,364 samples
  - Minority: 344
  - Majority: 199,020
Val: 42,721 samples
  - Minority: 74
  - Majority: 42,647
Test: 42,722 samples
  - Minority: 74
  - Majority: 42,648


In [22]:
# Precompute case-control distances (required for k-center matching)
# This follows the pattern from kcenter_hyperparameter_search_global.py

from precompute_distances import compute_distances_batched, save_distances_hdf5
import model_pipeline
import importlib    
importlib.reload(model_pipeline)
from model_pipeline import get_preprocessor
from sklearn.impute import SimpleImputer

def precompute_case_control_distances(
    train_df, 
    target_col,
    feature_cols,
    cat_columns,
    true_num_columns,
    dataset_name,
    seed=123
):
    """
    Precompute case-control distances and save to H5 file.
    This is required before running k-center matching.
    """
    print(f"\n{'='*80}")
    print(f"PRECOMPUTING CASE-CONTROL DISTANCES FOR {dataset_name.upper()}")
    print(f"{'='*80}\n")
    
    # Extract cases (minority) and controls (majority)
    cases = train_df[train_df[target_col] == 1].copy()
    controls = train_df[train_df[target_col] == 0].copy()
    
    n_cases = len(cases)
    n_controls = len(controls)
    
    print(f"Cases (minority): {n_cases:,}")
    print(f"Controls (majority): {n_controls:,}")
    
    # Prepare feature matrices for cases and controls
    X_cases = cases[feature_cols].copy()
    X_controls = controls[feature_cols].copy()
    
    # Handle missing values
    numeric_cols = X_cases.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X_cases.select_dtypes(include=["object", "category"]).columns.tolist()
    
    if numeric_cols:
        imputer = SimpleImputer(strategy='median')
        X_cases[numeric_cols] = imputer.fit_transform(X_cases[numeric_cols])
        X_controls[numeric_cols] = imputer.transform(X_controls[numeric_cols])
    
    if categorical_cols:
        cat_imputer = SimpleImputer(strategy="most_frequent")
        X_cases[categorical_cols] = cat_imputer.fit_transform(X_cases[categorical_cols])
        X_controls[categorical_cols] = cat_imputer.transform(X_controls[categorical_cols])
    
    # Preprocess features
    print(f"\nPreprocessing features...")
    preprocessor = get_preprocessor(
        df=X_controls,  # Fit on controls
        categorical_cols=cat_columns,
        numeric_cols=true_num_columns,
        verbose=True
    )
    
    # Fit preprocessor on controls, then transform both cases and controls
    X_controls_processed = preprocessor.fit_transform(X_controls)
    X_cases_processed = preprocessor.transform(X_cases)
    
    print(f"  Cases processed shape: {X_cases_processed.shape}")
    print(f"  Controls processed shape: {X_controls_processed.shape}")
    
    # Compute distances
    print(f"\nComputing distances (this may take a while)...")
    distances = compute_distances_batched(
        X_controls_processed,
        X_cases_processed,
        batch_size=1000,
        dtype=np.float32
    )
    
    print(f"  Distance matrix shape: {distances.shape}")
    print(f"  Distance range: [{distances.min():.3f}, {distances.max():.3f}]")
    print(f"  Distance mean: {distances.mean():.3f}")
    
    # Save to H5
    output_dir = "./precomputed_distances"
    os.makedirs(output_dir, exist_ok=True)
    h5_path = os.path.join(output_dir, f"distances_{dataset_name}_seed_{seed}.h5")
    
    majority_enrolids = controls["ENROLID"].to_numpy()
    minority_enrolids = cases["ENROLID"].to_numpy()
    
    save_distances_hdf5(
        distances,
        majority_enrolids,
        minority_enrolids,
        h5_path,
        compression='gzip'
    )
    
    print(f"\n✓ Saved distances to: {h5_path}")
    
    return h5_path, X_controls_processed, majority_enrolids

# Precompute distances for oil dataset
h5_path_oil, X_controls_oil, majority_enrolids_oil = precompute_case_control_distances(
    train_oil, 'target', feature_cols_oil, CAT_COLUMNS_OIL, TRUE_NUM_COLUMNS_OIL, 
    'oil', seed=TRAIN_TEST_SEED
)


PRECOMPUTING CASE-CONTROL DISTANCES FOR OIL

Cases (minority): 29
Controls (majority): 626

Preprocessing features...
→ Building preprocessor:
   • OneHotEncoder on: []
   • StandardScaler on: ['attr1', 'attr2', 'attr3', 'attr4', 'attr5', 'attr6', 'attr7', 'attr8', 'attr9', 'attr10', 'attr11', 'attr12', 'attr13', 'attr14', 'attr15', 'attr16', 'attr17', 'attr18', 'attr19', 'attr20', 'attr21', 'attr22', 'attr24', 'attr25', 'attr26', 'attr27', 'attr28', 'attr29', 'attr30', 'attr31', 'attr32', 'attr33', 'attr34', 'attr35', 'attr36', 'attr37', 'attr38', 'attr39', 'attr40', 'attr41', 'attr42', 'attr43', 'attr44', 'attr45', 'attr47', 'attr48', 'attr49']
  Cases processed shape: (29, 49)
  Controls processed shape: (626, 49)

Computing distances (this may take a while)...
Computing 626 x 29 = 18,154 distances
Output size: 0.1 MB (<class 'numpy.float32'>)


Computing distances: 100%|██████████| 1/1 [00:00<00:00, 825.65it/s]

  Distance matrix shape: (626, 29)
  Distance range: [1.151, 52.907]
  Distance mean: 9.493

Saving to HDF5: ./precomputed_distances/distances_oil_seed_123.h5
  ✓ Saved 0.1 MB

✓ Saved distances to: ./precomputed_distances/distances_oil_seed_123.h5


In [23]:
# Precompute distances for credit card fraud dataset
if X_fraud is not None:
    h5_path_fraud, X_controls_fraud, majority_enrolids_fraud = precompute_case_control_distances(
        train_fraud, 'target', feature_cols_fraud, CAT_COLUMNS_FRAUD, TRUE_NUM_COLUMNS_FRAUD,
        'creditcard_fraud', seed=TRAIN_TEST_SEED
    )
else:
    print("\n⚠️  Skipping credit card fraud distance computation (file not loaded)")


PRECOMPUTING CASE-CONTROL DISTANCES FOR CREDITCARD_FRAUD

Cases (minority): 344
Controls (majority): 199,020

Preprocessing features...
→ Building preprocessor:
   • OneHotEncoder on: []
   • StandardScaler on: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']
  Cases processed shape: (344, 30)
  Controls processed shape: (199020, 30)

Computing distances (this may take a while)...
Computing 199,020 x 344 = 68,462,880 distances
Output size: 273.9 MB (<class 'numpy.float32'>)


Computing distances: 100%|██████████| 200/200 [00:00<00:00, 1034.57it/s]


  Distance matrix shape: (199020, 344)
  Distance range: [0.178, 240.310]
  Distance mean: 24.092

Saving to HDF5: ./precomputed_distances/distances_creditcard_fraud_seed_123.h5
  ✓ Saved 241.0 MB

✓ Saved distances to: ./precomputed_distances/distances_creditcard_fraud_seed_123.h5


In [26]:
# Summary: Both datasets are now ready for k-center matching!
print("\n" + "="*80)
print("DATASETS READY FOR K-CENTER MATCHING")
print("="*80)

print("\nOIL DATASET:")
print(f"  Train size: {len(train_oil):,}")
print(f"  Feature columns: {len(feature_cols_oil)}")
print(f"  Distance file: {h5_path_oil}")
print(f"  Target column: 'target'")

if X_fraud is not None:
    print("\nCREDIT CARD FRAUD DATASET:")
    print(f"  Train size: {len(train_fraud):,}")
    print(f"  Feature columns: {len(feature_cols_fraud)}")
    print(f"  Distance file: {h5_path_fraud}")
    print(f"  Target column: 'target'")
    print(f"  Minority class (fraud): {(train_fraud['target'] == 1).sum():,}")
    print(f"  Majority class (normal): {(train_fraud['target'] == 0).sum():,}")
else:
    print("\nCREDIT CARD FRAUD DATASET:")
    print("  ⚠️  Not loaded - please download from Kaggle and update fraud_path")

print("\n" + "="*80)
print("Next steps:")
print("1. Run run_global_kcenter_matching() using train_oil/df_oil or train_fraud/df_fraud")
print("2. Use feature_cols_oil/CAT_COLUMNS_OIL/TRUE_NUM_COLUMNS_OIL or *_fraud equivalents")
print("3. Use h5_path_oil or h5_path_fraud for pn_h5_path")
print("="*80)


DATASETS READY FOR K-CENTER MATCHING

OIL DATASET:
  Train size: 655
  Feature columns: 49
  Distance file: ./precomputed_distances/distances_oil_seed_123.h5
  Target column: 'target'

CREDIT CARD FRAUD DATASET:
  Train size: 199,364
  Feature columns: 30
  Distance file: ./precomputed_distances/distances_creditcard_fraud_seed_123.h5
  Target column: 'target'
  Minority class (fraud): 344
  Majority class (normal): 199,020

Next steps:
1. Run run_global_kcenter_matching() using train_oil/df_oil or train_fraud/df_fraud
2. Use feature_cols_oil/CAT_COLUMNS_OIL/TRUE_NUM_COLUMNS_OIL or *_fraud equivalents
3. Use h5_path_oil or h5_path_fraud for pn_h5_path


In [27]:
# Fix ENROLID mismatch issue
# The error occurs because precomputed distance files have ENROLIDs that don't match
# the current training data. We need to delete old distance files and recompute them.

import os
import h5py
import numpy as np

def check_and_fix_enrolid_mismatch(dataset_name, h5_path, train_df, dnn_dir):
    """
    Check if ENROLIDs in distance files match training data, and fix if needed.
    """
    print(f"\n{'='*80}")
    print(f"CHECKING ENROLID CONSISTENCY FOR {dataset_name.upper()}")
    print(f"{'='*80}\n")
    
    # Check H5 file ENROLIDs
    if os.path.exists(h5_path):
        with h5py.File(h5_path, 'r') as f:
            h5_majority_ids = f['majority_enrolids'][:]
            h5_minority_ids = f['minority_enrolids'][:]
        
        train_controls = train_df[train_df['target'] == 0]['ENROLID'].values
        train_cases = train_df[train_df['target'] == 1]['ENROLID'].values
        
        h5_maj_set = set(h5_majority_ids.astype(int))
        train_controls_set = set(train_controls.astype(int))
        
        print(f"H5 file majority ENROLIDs: {len(h5_majority_ids):,}")
        print(f"Training controls ENROLIDs: {len(train_controls):,}")
        print(f"Match: {h5_maj_set == train_controls_set}")
        
        if h5_maj_set != train_controls_set:
            print(f"\n⚠️  ENROLID MISMATCH DETECTED!")
            print(f"   H5 has {len(h5_maj_set - train_controls_set)} ENROLIDs not in training")
            print(f"   Training has {len(train_controls_set - h5_maj_set)} ENROLIDs not in H5")
            print(f"\n   Solution: Delete and recompute distance files")
            return False
    else:
        print(f"H5 file not found: {h5_path}")
        return False
    
    # Check DNN files
    dnn_matrix_path = os.path.join(dnn_dir, "leaf_global_dnn_matrix.npy")
    dnn_enrolids_path = os.path.join(dnn_dir, "leaf_global_dnn_enrolids.npy")
    
    if os.path.exists(dnn_enrolids_path):
        dnn_ids = np.load(dnn_enrolids_path)
        dnn_set = set(dnn_ids.astype(int))
        
        print(f"\nDNN file ENROLIDs: {len(dnn_ids):,}")
        print(f"Match with training: {dnn_set == train_controls_set}")
        
        if dnn_set != train_controls_set:
            print(f"\n⚠️  DNN ENROLID MISMATCH DETECTED!")
            print(f"   DNN has {len(dnn_set - train_controls_set)} ENROLIDs not in training")
            print(f"   Training has {len(train_controls_set - dnn_set)} ENROLIDs not in DNN")
            return False
    
    print(f"\n✓ All ENROLIDs match!")
    return True

# Check oil dataset
print("Checking oil dataset...")
dnn_dir_oil = f"./precomputed_distances/global_dnn_seed_{TRAIN_TEST_SEED}"
oil_ok = check_and_fix_enrolid_mismatch("oil", h5_path_oil, train_oil, dnn_dir_oil)

# Check fraud dataset if loaded
if 'X_fraud' in globals() and X_fraud is not None:
    print("\n" + "="*80)
    print("Checking credit card fraud dataset...")
    fraud_ok = check_and_fix_enrolid_mismatch("fraud", h5_path_fraud, train_fraud, dnn_dir_oil)
else:
    fraud_ok = True

print("\n" + "="*80)
print("RECOMMENDATION:")
print("="*80)
if not oil_ok or not fraud_ok:
    print("⚠️  ENROLID mismatches detected!")
    print("\nTo fix:")
    print("1. Delete the old distance files:")
    print(f"   - {h5_path_oil}")
    if 'h5_path_fraud' in globals():
        print(f"   - {h5_path_fraud}")
    print(f"   - {dnn_dir_oil}/leaf_global_dnn_matrix.npy")
    print(f"   - {dnn_dir_oil}/leaf_global_dnn_enrolids.npy")
    print("\n2. Re-run the distance precomputation cells (Cells 5-6)")
    print("3. Then re-run the k-center matching cell")
else:
    print("✓ All ENROLIDs are consistent. You can proceed with k-center matching.")

Checking oil dataset...

CHECKING ENROLID CONSISTENCY FOR OIL

H5 file majority ENROLIDs: 626
Training controls ENROLIDs: 626
Match: True

DNN file ENROLIDs: 22,819
Match with training: False

⚠️  DNN ENROLID MISMATCH DETECTED!
   DNN has 22819 ENROLIDs not in training
   Training has 626 ENROLIDs not in DNN

Checking credit card fraud dataset...

CHECKING ENROLID CONSISTENCY FOR FRAUD

H5 file majority ENROLIDs: 199,020
Training controls ENROLIDs: 199,020
Match: True

DNN file ENROLIDs: 22,819
Match with training: False

⚠️  DNN ENROLID MISMATCH DETECTED!
   DNN has 22819 ENROLIDs not in training
   Training has 199020 ENROLIDs not in DNN

RECOMMENDATION:
⚠️  ENROLID mismatches detected!

To fix:
1. Delete the old distance files:
   - ./precomputed_distances/distances_oil_seed_123.h5
   - ./precomputed_distances/distances_creditcard_fraud_seed_123.h5
   - ./precomputed_distances/global_dnn_seed_123/leaf_global_dnn_matrix.npy
   - ./precomputed_distances/global_dnn_seed_123/leaf_global

In [ ]:
# Delete old distance files to force recomputation with correct ENROLIDs
# Run this cell if you got ENROLID mismatch errors

import shutil

def delete_distance_files(dataset_name, h5_path, dnn_dir):
    """Delete distance files to force recomputation."""
    print(f"\nDeleting distance files for {dataset_name}...")
    
    deleted = []
    
    # Delete H5 file
    if os.path.exists(h5_path):
        os.remove(h5_path)
        deleted.append(h5_path)
        print(f"  ✓ Deleted: {h5_path}")
    
    # Delete DNN files
    dnn_matrix_path = os.path.join(dnn_dir, "leaf_global_dnn_matrix.npy")
    dnn_enrolids_path = os.path.join(dnn_dir, "leaf_global_dnn_enrolids.npy")
    
    if os.path.exists(dnn_matrix_path):
        os.remove(dnn_matrix_path)
        deleted.append(dnn_matrix_path)
        print(f"  ✓ Deleted: {dnn_matrix_path}")
    
    if os.path.exists(dnn_enrolids_path):
        os.remove(dnn_enrolids_path)
        deleted.append(dnn_enrolids_path)
        print(f"  ✓ Deleted: {dnn_enrolids_path}")
    
    if not deleted:
        print(f"  No files to delete (they don't exist)")
    else:
        print(f"\n✓ Deleted {len(deleted)} file(s). Now recompute distances in Cells 5-6.")
    
    return len(deleted)

# Uncomment the lines below to delete files for a specific dataset
# This will force recomputation with the correct ENROLIDs

# For oil dataset:
# delete_distance_files("oil", h5_path_oil, dnn_dir_oil)

# For fraud dataset (if loaded):
# if 'h5_path_fraud' in globals():
#     delete_distance_files("fraud", h5_path_fraud, dnn_dir_oil)

print("="*80)
print("TO FIX ENROLID MISMATCH:")
print("="*80)
print("1. Uncomment the delete_distance_files() calls above")
print("2. Run this cell to delete old files")
print("3. Re-run the distance precomputation cells (Cells 5-6)")
print("4. Then re-run the k-center matching cell (Cell 10)")
print("="*80)

In [30]:
# Run k-center matching on prepared datasets
# This follows the same pattern as kcenter_hyperparameter_search_global.py
import kcenter_hyperparameter_search_global
importlib.reload(kcenter_hyperparameter_search_global)
from kcenter_hyperparameter_search_global import run_global_kcenter_matching

print("="*80)
print("RUNNING K-CENTER MATCHING")
print("="*80)

# ============================================================================
# OIL DATASET
# ============================================================================
print("\n" + "="*80)
print("OIL DATASET - K-CENTER MATCHING")
print("="*80)

# Use dataset-specific DNN directory to avoid conflicts
dnn_dir_oil = f"./precomputed_distances/global_dnn_oil_seed_{TRAIN_TEST_SEED}"

matching_result_oil = run_global_kcenter_matching(
    train_pd=train_oil,
    target_col='target',
    feature_cols=feature_cols_oil,
    pn_h5_path=h5_path_oil,
    matching_ratio=1,  # 1:1 matching
    case_weighting=None,  # or "boundary", "uncertainty", "density_inverse"
    use_adaptive_pool=True,
    seed_method="smart",  # or "centroid", "density", "random"
    CAT_COLUMNS=CAT_COLUMNS_OIL,
    TRUE_NUM_COLUMNS=TRUE_NUM_COLUMNS_OIL,
    COST_COLUMNS=None,  # No cost columns in UCI datasets
    dnn_out_dir=dnn_dir_oil,  # Dataset-specific DNN directory
)

print("\n✓ Oil dataset matching complete!")
print(f"  Selected controls: {len(matching_result_oil['selected_control_enrolids']):,}")
print(f"  Unique controls: {len(set(matching_result_oil['selected_control_enrolids'])):,}")
if len(matching_result_oil['match_costs']) > 0:
    print(f"  Mean matching cost: {matching_result_oil['match_costs'].mean():.4f}")

# ============================================================================
# CREDIT CARD FRAUD DATASET
# ============================================================================
if 'X_fraud' in globals() and X_fraud is not None:
    print("\n" + "="*80)
    print("CREDIT CARD FRAUD DATASET - K-CENTER MATCHING")
    print("="*80)
    
    # Use dataset-specific DNN directory to avoid conflicts
    dnn_dir_fraud = f"./precomputed_distances/global_dnn_fraud_seed_{TRAIN_TEST_SEED}"
    
    matching_result_fraud = run_global_kcenter_matching(
        train_pd=train_fraud,
        target_col='target',
        feature_cols=feature_cols_fraud,
        pn_h5_path=h5_path_fraud,
        matching_ratio=1,  # 1:1 matching
        case_weighting=None,  # or "boundary", "uncertainty", "density_inverse"
        use_adaptive_pool=True,
        seed_method="smart",  # or "centroid", "density", "random"
        CAT_COLUMNS=CAT_COLUMNS_FRAUD,
        TRUE_NUM_COLUMNS=TRUE_NUM_COLUMNS_FRAUD,
        COST_COLUMNS=None,  # No cost columns in UCI datasets
        dnn_out_dir=dnn_dir_fraud,  # Dataset-specific DNN directory
    )
    
    print("\n✓ Credit card fraud dataset matching complete!")
    print(f"  Selected controls: {len(matching_result_fraud['selected_control_enrolids']):,}")
    print(f"  Unique controls: {len(set(matching_result_fraud['selected_control_enrolids'])):,}")
    if len(matching_result_fraud['match_costs']) > 0:
        print(f"  Mean matching cost: {matching_result_fraud['match_costs'].mean():.4f}")
else:
    print("\n⚠️  Skipping credit card fraud dataset matching (dataset not loaded)")

print("\n" + "="*80)
print("K-CENTER MATCHING COMPLETE")
print("="*80)
print("\nMatching results are stored in:")
print("  - matching_result_oil: Oil dataset results")
if 'X_fraud' in globals() and X_fraud is not None:
    print("  - matching_result_fraud: Credit card fraud dataset results")
print("\nEach result contains:")
print("  - selected_control_enrolids: matched control IDs")
print("  - match_costs: matching costs")
print("  - case_to_control_map: mapping from cases to controls")
print("  - n_cases_total, n_cases_matched, n_controls, M, etc.")

RUNNING K-CENTER MATCHING

OIL DATASET - K-CENTER MATCHING

GLOBAL K-CENTER MATCHING CONFIGURATION:
  case_weighting: None
  use_adaptive_pool: True
  seed_method: smart
  matching_ratio: 1:1

Global Statistics:
  Cases (minority): 29
  Controls (majority): 626
  Ratio: 21.59:1

  Preprocessing:
    Features: 49
    Preprocessed shape: (626, 49)

  Preparing control-control distances (global)...
    ⚙️  Computing global control-control distances (this is done once)...
[leaf global] computing d_nn for n=626 -> ~0.00 GB float32


leaf global d_nn: 100%|██████████| 1/1 [00:00<00:00, 168.56it/s]

[leaf global] saved:
  ./precomputed_distances/global_dnn_oil_seed_123/leaf_global_dnn_matrix.npy
  ./precomputed_distances/global_dnn_oil_seed_123/leaf_global_dnn_enrolids.npy
    ✓ Control-control distances computed and saved

  K-Center Configuration:
    M (candidate pool size): 626 / 626 (100.0%)
    Cases to match: 29
    Seed method: smart
    Adaptive pool: True
    Case weighting: None

  Running two-stage k-center matching (1:1)...
  Seed selection method: 'smart'
    Smart seed selected: index 368 (mean dist to cases: 7.3217)
  Auto-computed tau (95th percentile of best distances): 7.3158
  Adaptive pool stopped at 63 candidates (max cost: 5.1034)


    ✓ Matching complete!
    Cases matched: 29
    Total selected controls: 29 (unique: 29)
    Mean matching cost: 4.4373
    Sampling time: 0.0901s
    Memory used: 2458.2 MB (Δ +13.3 MB)

✓ Oil dataset matching complete!
  Selected controls: 29
  Unique controls: 29
  Mean matching cost: 4.4373

CREDIT CARD FRAUD DATASET - K-CENTER MATCHING

GLOBAL K-CENTER MATCHING CONFIGURATION:
  case_weighting: None
  use_adaptive_pool: True
  seed_method: smart
  matching_ratio: 1:1

Global Statistics:
  Cases (minority): 344
  Controls (majority): 199,020
  Ratio: 578.55:1

  Preprocessing:
    Features: 30
    Preprocessed shape: (199020, 30)

  Preparing control-control distances (global)...
    ⚙️  Computing global control-control distances (this is done once)...
[leaf global] computing d_nn for n=199,020 -> ~158.44 GB float32


leaf global d_nn: 100%|██████████| 266/266 [02:30<00:00,  1.76it/s]


[leaf global] saved:
  ./precomputed_distances/global_dnn_fraud_seed_123/leaf_global_dnn_matrix.npy
  ./precomputed_distances/global_dnn_fraud_seed_123/leaf_global_dnn_enrolids.npy
    ✓ Control-control distances computed and saved

  K-Center Configuration:
    M (candidate pool size): 8,000 / 199,020 (4.0%)
    Cases to match: 344
    Seed method: smart
    Adaptive pool: True
    Case weighting: None

  Running two-stage k-center matching (1:1)...
  Seed selection method: 'smart'
    Smart seed selected: index 64801 (mean dist to cases: 18.4375)
  Auto-computed tau (95th percentile of best distances): 24.7971
  Adaptive pool stopped at 8000 candidates (max cost: 9.6450)
    ✓ Matching complete!
    Cases matched: 344
    Total selected controls: 344 (unique: 344)
    Mean matching cost: 19.5473
    Sampling time: 5.91s
    Memory used: 3598.3 MB (Δ +450.0 MB)

✓ Credit card fraud dataset matching complete!
  Selected controls: 344
  Unique controls: 344
  Mean matching cost: 19.5473

In [32]:
# Train and evaluate OCT models on undersampled datasets
from kcenter_hyperparameter_search_global import build_undersampled_dataset
from model_IAI import finetune_oct, evaluate_binary_oct
import os

# OCT hyperparameters (same as kcenter_hyperparameter_search_global.py)
OCT_DEPTHS = [5,7]
OCT_MINBUCKETS = [10,25]
OCT_MINBUCKETS_FRAUD = [25, 50,75]
OCT_CPS = [0.00001, 0.0001, 0.001, 0.01]

# Results directory
RESULTS_DIR = "./uci_experiments_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("="*80)
print("TRAINING AND EVALUATING OCT MODELS")
print("="*80)

# ============================================================================
# OIL DATASET
# ============================================================================
print("\n" + "="*80)
print("OIL DATASET - OCT TRAINING")
print("="*80)

# Build undersampled dataset
undersampled_oil = build_undersampled_dataset(
    train_pd=train_oil,
    matching_result=matching_result_oil,
    target_col='target',
    matching_ratio=1,  # Match the ratio used in matching
)

# Prepare validation and test sets
X_val_oil = val_oil[feature_cols_oil]
y_val_oil = val_oil['target']
X_test_oil = test_oil[feature_cols_oil]
y_test_oil = test_oil['target']

# Train OCT
print(f"\n{'='*80}")
print("TRAINING OCT MODEL: Oil Dataset")
print(f"{'='*80}\n")

balanced_model_oil, balanced_params_oil, _, preprocessor_oil, feature_names_oil = finetune_oct(
    X_train=undersampled_oil[feature_cols_oil],
    y_train=undersampled_oil['target'],
    X_val=X_val_oil,
    y_val=y_val_oil,
    categorical_cols=CAT_COLUMNS_OIL,
    numeric_cols=TRUE_NUM_COLUMNS_OIL,
    depths=OCT_DEPTHS,
    minbuckets=OCT_MINBUCKETS,
    cps=OCT_CPS,
)

# Evaluate
metrics_oil = evaluate_binary_oct(
    balanced_model_oil, X_test_oil, y_test_oil, preprocessor_oil, feature_names_oil,
    results_dir=RESULTS_DIR, ratio=1.0
)

print(f"\n✓ Oil dataset OCT training complete!")
print(f"   Best params: {balanced_params_oil}")
if isinstance(metrics_oil, dict):
    print(f"   AUC: {metrics_oil.get('auc', 'N/A'):.4f}" if isinstance(metrics_oil.get('auc'), (int, float)) else f"   AUC: {metrics_oil.get('auc', 'N/A')}")
    print(f"   PR-AUC: {metrics_oil.get('pr_auc', 'N/A'):.4f}" if isinstance(metrics_oil.get('pr_auc'), (int, float)) else f"   PR-AUC: {metrics_oil.get('pr_auc', 'N/A')}")
    print(f"   Optimal F1: {metrics_oil.get('optimal_f1', 'N/A'):.4f}" if isinstance(metrics_oil.get('optimal_f1'), (int, float)) else f"   Optimal F1: {metrics_oil.get('optimal_f1', 'N/A')}")
    print(f"   Best MCC: {metrics_oil.get('best_mcc', 'N/A'):.4f}" if isinstance(metrics_oil.get('best_mcc'), (int, float)) else f"   Best MCC: {metrics_oil.get('best_mcc', 'N/A')}")

# ============================================================================
# CREDIT CARD FRAUD DATASET
# ============================================================================
if 'X_fraud' in globals() and X_fraud is not None and 'matching_result_fraud' in globals():
    print("\n" + "="*80)
    print("CREDIT CARD FRAUD DATASET - OCT TRAINING")
    print("="*80)
    
    # Build undersampled dataset
    undersampled_fraud = build_undersampled_dataset(
        train_pd=train_fraud,
        matching_result=matching_result_fraud,
        target_col='target',
        matching_ratio=1,  # Match the ratio used in matching
    )
    
    # Prepare validation and test sets
    X_val_fraud = val_fraud[feature_cols_fraud]
    y_val_fraud = val_fraud['target']
    X_test_fraud = test_fraud[feature_cols_fraud]
    y_test_fraud = test_fraud['target']
    
    # Train OCT
    print(f"\n{'='*80}")
    print("TRAINING OCT MODEL: Credit Card Fraud Dataset")
    print(f"{'='*80}\n")
    
    balanced_model_fraud, balanced_params_fraud, _, preprocessor_fraud, feature_names_fraud = finetune_oct(
        X_train=undersampled_fraud[feature_cols_fraud],
        y_train=undersampled_fraud['target'],
        X_val=X_val_fraud,
        y_val=y_val_fraud,
        categorical_cols=CAT_COLUMNS_FRAUD,
        numeric_cols=TRUE_NUM_COLUMNS_FRAUD,
        depths=OCT_DEPTHS,
        minbuckets=OCT_MINBUCKETS_FRAUD,
        cps=OCT_CPS,
    )
    
    # Evaluate
    metrics_fraud = evaluate_binary_oct(
        balanced_model_fraud, X_test_fraud, y_test_fraud, preprocessor_fraud, feature_names_fraud,
        results_dir=RESULTS_DIR, ratio=1.0
    )
    
    print(f"\n✓ Credit card fraud dataset OCT training complete!")
    print(f"   Best params: {balanced_params_fraud}")
    if isinstance(metrics_fraud, dict):
        print(f"   AUC: {metrics_fraud.get('auc', 'N/A'):.4f}" if isinstance(metrics_fraud.get('auc'), (int, float)) else f"   AUC: {metrics_fraud.get('auc', 'N/A')}")
        print(f"   PR-AUC: {metrics_fraud.get('pr_auc', 'N/A'):.4f}" if isinstance(metrics_fraud.get('pr_auc'), (int, float)) else f"   PR-AUC: {metrics_fraud.get('pr_auc', 'N/A')}")
        print(f"   Optimal F1: {metrics_fraud.get('optimal_f1', 'N/A'):.4f}" if isinstance(metrics_fraud.get('optimal_f1'), (int, float)) else f"   Optimal F1: {metrics_fraud.get('optimal_f1', 'N/A')}")
        print(f"   Best MCC: {metrics_fraud.get('best_mcc', 'N/A'):.4f}" if isinstance(metrics_fraud.get('best_mcc'), (int, float)) else f"   Best MCC: {metrics_fraud.get('best_mcc', 'N/A')}")
else:
    print("\n⚠️  Skipping credit card fraud dataset OCT training (matching not completed)")

print("\n" + "="*80)
print("OCT TRAINING AND EVALUATION COMPLETE")
print("="*80)
print(f"\nResults saved to: {RESULTS_DIR}/")
print("\nModels and metrics stored in:")
print("  - balanced_model_oil, balanced_params_oil, metrics_oil")
if 'X_fraud' in globals() and X_fraud is not None and 'matching_result_fraud' in globals():
    print("  - balanced_model_fraud, balanced_params_fraud, metrics_fraud")

TRAINING AND EVALUATING OCT MODELS

OIL DATASET - OCT TRAINING

BUILDING UNDERSAMPLED TRAINING DATASET

✓ Collected all minority samples: 29
✓ Collected selected majority samples: 29

✓ Undersampled dataset created:
   Total samples: 58
   Minority: 29
   Majority: 29
   Ratio (maj:min): 1.00:1

Class distribution:
target
0    29
1    29
Name: count, dtype: int64

TRAINING OCT MODEL: Oil Dataset

Finetuning OCT with depths: [5, 7], minbuckets: [10, 25], cps: [1e-05, 0.0001, 0.001, 0.01], for best PR-AUC!!!
→ Building preprocessor:
   • OneHotEncoder on: []
   • StandardScaler on: ['attr1', 'attr2', 'attr3', 'attr4', 'attr5', 'attr6', 'attr7', 'attr8', 'attr9', 'attr10', 'attr11', 'attr12', 'attr13', 'attr14', 'attr15', 'attr16', 'attr17', 'attr18', 'attr19', 'attr20', 'attr21', 'attr22', 'attr24', 'attr25', 'attr26', 'attr27', 'attr28', 'attr29', 'attr30', 'attr31', 'attr32', 'attr33', 'attr34', 'attr35', 'attr36', 'attr37', 'attr38', 'attr39', 'attr40', 'attr41', 'attr42', 'attr43', '

In [ ]:
# Train OCT on original imbalanced training sets (baseline comparison)
from model_IAI import finetune_oct, evaluate_binary_oct

print("="*80)
print("TRAINING OCT ON ORIGINAL IMBALANCED DATASETS (BASELINE)")
print("="*80)

# ============================================================================
# OIL DATASET - BASELINE
# ============================================================================
print("\n" + "="*80)
print("OIL DATASET - BASELINE (ORIGINAL IMBALANCED)")
print("="*80)

# Prepare validation and test sets (same as before)
X_val_oil = val_oil[feature_cols_oil]
y_val_oil = val_oil['target']
X_test_oil = test_oil[feature_cols_oil]
y_test_oil = test_oil['target']

# Prepare original imbalanced training set
X_train_oil_imbalanced = train_oil[feature_cols_oil]
y_train_oil_imbalanced = train_oil['target']

print(f"\nOriginal training set distribution:")
print(f"  Total samples: {len(X_train_oil_imbalanced):,}")
print(f"  Minority (1): {y_train_oil_imbalanced.sum():,} ({y_train_oil_imbalanced.mean()*100:.2f}%)")
print(f"  Majority (0): {(y_train_oil_imbalanced == 0).sum():,} ({(1-y_train_oil_imbalanced.mean())*100:.2f}%)")
print(f"  Imbalance ratio: {(y_train_oil_imbalanced == 0).sum() / y_train_oil_imbalanced.sum():.2f}:1")

# Train OCT on imbalanced data
print(f"\n{'='*80}")
print("TRAINING OCT MODEL: Oil Dataset (Imbalanced Baseline)")
print(f"{'='*80}\n")

baseline_model_oil, baseline_params_oil, _, baseline_preprocessor_oil, baseline_feature_names_oil = finetune_oct(
    X_train=X_train_oil_imbalanced,
    y_train=y_train_oil_imbalanced,
    X_val=X_val_oil,
    y_val=y_val_oil,
    categorical_cols=CAT_COLUMNS_OIL,
    numeric_cols=TRUE_NUM_COLUMNS_OIL,
    depths=OCT_DEPTHS,
    minbuckets=OCT_MINBUCKETS,
    cps=OCT_CPS,
)

# Evaluate
baseline_metrics_oil = evaluate_binary_oct(
    baseline_model_oil, X_test_oil, y_test_oil, baseline_preprocessor_oil, baseline_feature_names_oil,
    results_dir=RESULTS_DIR, ratio=1.0
)

print(f"\n✓ Oil dataset baseline OCT training complete!")
print(f"   Best params: {baseline_params_oil}")
if isinstance(baseline_metrics_oil, dict):
    print(f"   AUC: {baseline_metrics_oil.get('auc', 'N/A'):.4f}" if isinstance(baseline_metrics_oil.get('auc'), (int, float)) else f"   AUC: {baseline_metrics_oil.get('auc', 'N/A')}")
    print(f"   PR-AUC: {baseline_metrics_oil.get('pr_auc', 'N/A'):.4f}" if isinstance(baseline_metrics_oil.get('pr_auc'), (int, float)) else f"   PR-AUC: {baseline_metrics_oil.get('pr_auc', 'N/A')}")
    print(f"   Optimal F1: {baseline_metrics_oil.get('optimal_f1', 'N/A'):.4f}" if isinstance(baseline_metrics_oil.get('optimal_f1'), (int, float)) else f"   Optimal F1: {baseline_metrics_oil.get('optimal_f1', 'N/A')}")
    print(f"   Best MCC: {baseline_metrics_oil.get('best_mcc', 'N/A'):.4f}" if isinstance(baseline_metrics_oil.get('best_mcc'), (int, float)) else f"   Best MCC: {baseline_metrics_oil.get('best_mcc', 'N/A')}")

# ============================================================================
# CREDIT CARD FRAUD DATASET - BASELINE
# ============================================================================
if 'X_fraud' in globals() and X_fraud is not None:
    print("\n" + "="*80)
    print("CREDIT CARD FRAUD DATASET - BASELINE (ORIGINAL IMBALANCED)")
    print("="*80)
    
    # Prepare validation and test sets
    X_val_fraud = val_fraud[feature_cols_fraud]
    y_val_fraud = val_fraud['target']
    X_test_fraud = test_fraud[feature_cols_fraud]
    y_test_fraud = test_fraud['target']
    
    # Prepare original imbalanced training set
    X_train_fraud_imbalanced = train_fraud[feature_cols_fraud]
    y_train_fraud_imbalanced = train_fraud['target']
    
    print(f"\nOriginal training set distribution:")
    print(f"  Total samples: {len(X_train_fraud_imbalanced):,}")
    print(f"  Minority (1): {y_train_fraud_imbalanced.sum():,} ({y_train_fraud_imbalanced.mean()*100:.2f}%)")
    print(f"  Majority (0): {(y_train_fraud_imbalanced == 0).sum():,} ({(1-y_train_fraud_imbalanced.mean())*100:.2f}%)")
    print(f"  Imbalance ratio: {(y_train_fraud_imbalanced == 0).sum() / y_train_fraud_imbalanced.sum():.2f}:1")
    
    # Train OCT on imbalanced data
    print(f"\n{'='*80}")
    print("TRAINING OCT MODEL: Credit Card Fraud Dataset (Imbalanced Baseline)")
    print(f"{'='*80}\n")
    
    baseline_model_fraud, baseline_params_fraud, _, baseline_preprocessor_fraud, baseline_feature_names_fraud = finetune_oct(
        X_train=X_train_fraud_imbalanced,
        y_train=y_train_fraud_imbalanced,
        X_val=X_val_fraud,
        y_val=y_val_fraud,
        categorical_cols=CAT_COLUMNS_FRAUD,
        numeric_cols=TRUE_NUM_COLUMNS_FRAUD,
        depths=OCT_DEPTHS,
        minbuckets=OCT_MINBUCKETS,
        cps=OCT_CPS,
    )
    
    # Evaluate
    baseline_metrics_fraud = evaluate_binary_oct(
        baseline_model_fraud, X_test_fraud, y_test_fraud, baseline_preprocessor_fraud, baseline_feature_names_fraud,
        results_dir=RESULTS_DIR, ratio=1.0
    )
    
    print(f"\n✓ Credit card fraud dataset baseline OCT training complete!")
    print(f"   Best params: {baseline_params_fraud}")
    if isinstance(baseline_metrics_fraud, dict):
        print(f"   AUC: {baseline_metrics_fraud.get('auc', 'N/A'):.4f}" if isinstance(baseline_metrics_fraud.get('auc'), (int, float)) else f"   AUC: {baseline_metrics_fraud.get('auc', 'N/A')}")
        print(f"   PR-AUC: {baseline_metrics_fraud.get('pr_auc', 'N/A'):.4f}" if isinstance(baseline_metrics_fraud.get('pr_auc'), (int, float)) else f"   PR-AUC: {baseline_metrics_fraud.get('pr_auc', 'N/A')}")
        print(f"   Optimal F1: {baseline_metrics_fraud.get('optimal_f1', 'N/A'):.4f}" if isinstance(baseline_metrics_fraud.get('optimal_f1'), (int, float)) else f"   Optimal F1: {baseline_metrics_fraud.get('optimal_f1', 'N/A')}")
        print(f"   Best MCC: {baseline_metrics_fraud.get('best_mcc', 'N/A'):.4f}" if isinstance(baseline_metrics_fraud.get('best_mcc'), (int, float)) else f"   Best MCC: {baseline_metrics_fraud.get('best_mcc', 'N/A')}")
else:
    print("\n⚠️  Skipping credit card fraud dataset baseline training (dataset not loaded)")

print("\n" + "="*80)
print("BASELINE OCT TRAINING COMPLETE")
print("="*80)
print(f"\nBaseline models and metrics stored in:")
print("  - baseline_model_oil, baseline_params_oil, baseline_metrics_oil")
if 'X_fraud' in globals() and X_fraud is not None:
    print("  - baseline_model_fraud, baseline_params_fraud, baseline_metrics_fraud")
print("\nCompare with k-center matching results:")
print("  - balanced_model_oil, balanced_params_oil, metrics_oil")
if 'X_fraud' in globals() and X_fraud is not None and 'matching_result_fraud' in globals():
    print("  - balanced_model_fraud, balanced_params_fraud, metrics_fraud")

TRAINING OCT ON ORIGINAL IMBALANCED DATASETS (BASELINE)

OIL DATASET - BASELINE (ORIGINAL IMBALANCED)

Original training set distribution:
  Total samples: 655
  Minority (1): 29 (4.43%)
  Majority (0): 626 (95.57%)
  Imbalance ratio: 21.59:1

TRAINING OCT MODEL: Oil Dataset (Imbalanced Baseline)

Finetuning OCT with depths: [5, 7], minbuckets: [10, 25], cps: [1e-05, 0.0001, 0.001, 0.01], for best PR-AUC!!!
→ Building preprocessor:
   • OneHotEncoder on: []
   • StandardScaler on: ['attr1', 'attr2', 'attr3', 'attr4', 'attr5', 'attr6', 'attr7', 'attr8', 'attr9', 'attr10', 'attr11', 'attr12', 'attr13', 'attr14', 'attr15', 'attr16', 'attr17', 'attr18', 'attr19', 'attr20', 'attr21', 'attr22', 'attr24', 'attr25', 'attr26', 'attr27', 'attr28', 'attr29', 'attr30', 'attr31', 'attr32', 'attr33', 'attr34', 'attr35', 'attr36', 'attr37', 'attr38', 'attr39', 'attr40', 'attr41', 'attr42', 'attr43', 'attr44', 'attr45', 'attr47', 'attr48', 'attr49']
Best params: (7, 10, 1e-05) @ PR-AUC: 0.251
Test dat